# 🎨 W Collection - Ürün Açıklaması Üretici

Florence-2 ile görsel analizi + LLM ile profesyonel ürün açıklaması

**Örnek Çıktı:**  
*"Yüzde yüz pamuk kumaştan üretilen kahve renkli W Collection kazak; düğmeli polo yakaya sahiptir."*

## 📦 Kurulum

In [ ]:
# Gerekli paketleri kur
!pip install -q transformers pillow google-generativeai anthropic openai einops timm

## 🔑 API Key Ayarları

**3 seçenek:**
1. **Google Gemini** (Colab'da ücretsiz) ✅ Önerilen
2. **Anthropic Claude** (En iyi kalite)
3. **OpenAI GPT** (Popüler alternatif)

In [ ]:
from google.colab import userdata

# API key'lerinizi Colab Secrets'a ekleyin
# Sol menü -> 🔑 Secrets -> Add new secret

# Hangisini kullanacaksanız onu seçin:
LLM_PROVIDER = "gemini"  # "gemini", "claude" veya "openai"

if LLM_PROVIDER == "gemini":
    import google.generativeai as genai
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')  # Colab Secrets'dan al
    genai.configure(api_key=GEMINI_API_KEY)
    
elif LLM_PROVIDER == "claude":
    import anthropic
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    
elif LLM_PROVIDER == "openai":
    from openai import OpenAI
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

print(f"✅ LLM Provider: {LLM_PROVIDER.upper()}")

## 🤖 Florence-2 Model Yükleme

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor
from PIL import Image
import requests
from io import BytesIO

model_id = "microsoft/Florence-2-base"

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32

print(f"🖥️  Device: {device} | dtype: {torch_dtype}")
print("📥 Florence-2 model yükleniyor...")

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    trust_remote_code=True,
).to(device)

processor = AutoProcessor.from_pretrained(
    model_id,
    trust_remote_code=True,
)

print("✅ Model ve processor hazır!")

## 🎯 Ana Fonksiyonlar

In [ ]:
def analyze_product_image(image_path_or_url):
    """
    Florence-2 ile ürün görselini analiz et
    
    Returns:
        dict: Görsel analiz sonuçları
    """
    # Görseli yükle (URL veya local path)
    if image_path_or_url.startswith('http'):
        response = requests.get(image_path_or_url)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = Image.open(image_path_or_url).convert("RGB")
    
    results = {}
    
    # 1. Detaylı Caption
    print("🔍 Detaylı görsel analizi yapılıyor...")
    prompt = "<MORE_DETAILED_CAPTION>"
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(device, torch_dtype)
    
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        num_beams=3,
    )
    
    output = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    results['detailed_caption'] = output.replace("</s>", "").replace("<MORE_DETAILED_CAPTION>", "").strip()
    
    # 2. Object Detection
    print("🔍 Nesneler tespit ediliyor...")
    prompt = "<OD>"
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(device, torch_dtype)
    
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        num_beams=3,
    )
    
    output = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    results['objects'] = output.replace("</s>", "").replace("<OD>", "").strip()
    
    # 3. Dense Region Caption
    print("🔍 Bölgesel detaylar analiz ediliyor...")
    prompt = "<DENSE_REGION_CAPTION>"
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(device, torch_dtype)
    
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        num_beams=3,
    )
    
    output = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    results['dense_captions'] = output.replace("</s>", "").replace("<DENSE_REGION_CAPTION>", "").strip()
    
    # 4. OCR
    print("🔍 Metin tespiti yapılıyor...")
    prompt = "<OCR>"
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(device, torch_dtype)
    
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        num_beams=3,
    )
    
    output = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    results['ocr_text'] = output.replace("</s>", "").replace("<OCR>", "").strip()
    
    print("✅ Görsel analizi tamamlandı!")
    
    return results, image

In [ ]:
def generate_product_description(visual_analysis, product_info=None):
    """
    LLM ile ürün açıklaması üret
    
    Args:
        visual_analysis: Florence-2'den gelen görsel analizi
        product_info: Ek ürün bilgileri (dict)
            - category: Ürün kategorisi
            - material: Malzeme bilgisi
            - color: Renk
            - size: Beden
    """
    
    # Prompt oluştur
    prompt = f"""Sen W Collection için ürün açıklamaları yazan bir e-ticaret içerik uzmanısın.

Aşağıdaki görsel analiz verilerine dayanarak, doğal ve akıcı bir Türkçe ürün açıklaması yaz.

## Görsel Analiz:
**Detaylı Açıklama:** {visual_analysis['detailed_caption']}

**Tespit Edilen Nesneler:** {visual_analysis['objects']}

**Bölgesel Detaylar:** {visual_analysis['dense_captions']}

**Görsel Üzerindeki Yazılar:** {visual_analysis['ocr_text']}
"""

    if product_info:
        prompt += "\n## Ek Ürün Bilgileri:\n"
        for key, value in product_info.items():
            if value:
                prompt += f"- {key}: {value}\n"
    
    prompt += """\n## Görev:
Yukarıdaki bilgilere dayanarak, W Collection için doğal ve akıcı bir ürün açıklaması yaz.

**Format:**
2-3 cümlelik, doğal akıcı Türkçe açıklama. 

**İyi Örnek:** "Yüzde yüz pamuk kumaştan üretilen kahve renkli W Collection kazak; düğmeli polo yakaya sahiptir. Günlük kullanım için idealdir."

**Kurallar:**
- Doğal ve akıcı Türkçe kullan
- Görsel analizden çıkardığın özellikleri dahil et (renk, stil, kesim, detaylar)
- Abartma yapma, sadece görsel ve verilen bilgilere dayanarak yaz
- Markdown, başlık veya bullet point kullanma - sadece düz metin
- W Collection markasını vurgula

Şimdi ürün açıklamasını yaz:
"""
    
    print("✍️  Ürün açıklaması oluşturuluyor...")
    
    # LLM'e gönder
    if LLM_PROVIDER == "gemini":
        model = genai.GenerativeModel('gemini-1.5-flash')
        response = model.generate_content(prompt)
        description = response.text.strip()
        
    elif LLM_PROVIDER == "claude":
        client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        response = client.messages.create(
            model="claude-sonnet-4-0",
            max_tokens=1024,
            messages=[{"role": "user", "content": prompt}]
        )
        description = response.content[0].text.strip()
        
    elif LLM_PROVIDER == "openai":
        client = OpenAI(api_key=OPENAI_API_KEY)
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}]
        )
        description = response.choices[0].message.content.strip()
    
    print("✅ Ürün açıklaması hazır!")
    
    return description

## 🚀 Kullanım - Tek Ürün

In [ ]:
# Görseli yükle (URL veya upload)
from google.colab import files
import matplotlib.pyplot as plt

# Seçenek 1: Google Drive'dan
# from google.colab import drive
# drive.mount('/content/drive')
# image_path = '/content/drive/MyDrive/product_image.jpg'

# Seçenek 2: Upload
print("📤 Lütfen ürün görselini yükleyin:")
uploaded = files.upload()
image_path = list(uploaded.keys())[0]

# Seçenek 3: URL
# image_path = "https://example.com/product.jpg"

print(f"\n📸 Görsel: {image_path}")

In [ ]:
# Görseli analiz et
visual_analysis, image = analyze_product_image(image_path)

# Görseli göster
plt.figure(figsize=(8, 8))
plt.imshow(image)
plt.axis('off')
plt.title('Ürün Görseli')
plt.show()

# Analiz sonuçlarını göster
print("\n" + "="*80)
print("GÖRSEL ANALİZ SONUÇLARI")
print("="*80)
for key, value in visual_analysis.items():
    print(f"\n📋 {key.upper()}:")
    print(f"   {value}")

In [ ]:
# Ürün açıklaması üret
# Opsiyonel: Ek bilgi ekleyebilirsiniz
product_info = {
    "Kategori": "kazak",
    "Malzeme": "%100 pamuk",
    "Renk": "kahverengi",
    # "Beden": "M",
    # "Koleksiyon": "2024 Sonbahar/Kış"
}

description = generate_product_description(visual_analysis, product_info)

print("\n" + "="*80)
print("✨ ÜRÜN AÇIKLAMASI")
print("="*80)
print(f"\n{description}\n")

## 🔄 Toplu İşlem - Çoklu Ürünler

In [ ]:
import os
import json
from tqdm import tqdm

def process_multiple_products(image_folder, output_file="product_descriptions.json"):
    """
    Klasördeki tüm görseller için açıklama üret
    """
    results = []
    
    # Görselleri bul
    image_extensions = {'.jpg', '.jpeg', '.png', '.webp'}
    image_files = [
        f for f in os.listdir(image_folder)
        if os.path.splitext(f)[1].lower() in image_extensions
    ]
    
    print(f"📁 Toplam {len(image_files)} görsel bulundu.\n")
    
    for image_file in tqdm(image_files, desc="İşleniyor"):
        image_path = os.path.join(image_folder, image_file)
        
        try:
            # Analiz
            visual_analysis, _ = analyze_product_image(image_path)
            
            # Açıklama
            description = generate_product_description(visual_analysis)
            
            results.append({
                "image_name": image_file,
                "description": description,
                "visual_analysis": visual_analysis,
                "status": "success"
            })
            
            print(f"✓ {image_file}")
            
        except Exception as e:
            print(f"✗ {image_file}: {str(e)}")
            results.append({
                "image_name": image_file,
                "error": str(e),
                "status": "failed"
            })
    
    # Sonuçları kaydet
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    
    print(f"\n✅ Sonuçlar kaydedildi: {output_file}")
    
    # Özet
    success = sum(1 for r in results if r['status'] == 'success')
    print(f"\n📊 Özet: {success}/{len(results)} başarılı")
    
    return results

In [ ]:
# Toplu işlem örneği
# image_folder = "/content/drive/MyDrive/product_images"
# results = process_multiple_products(image_folder, "w_collection_descriptions.json")

## 💾 Sonuçları İndirme

In [ ]:
# Sonuçları JSON olarak indir
# from google.colab import files
# files.download('w_collection_descriptions.json')

## 🎯 Hızlı Test Fonksiyonu

In [ ]:
def quick_describe(image_path_or_url, **product_info):
    """
    Tek satırda ürün açıklaması üret
    
    Örnek:
    quick_describe(
        "product.jpg",
        Kategori="kazak",
        Malzeme="%100 pamuk",
        Renk="kahverengi"
    )
    """
    visual_analysis, image = analyze_product_image(image_path_or_url)
    
    # Görseli göster
    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.axis('off')
    plt.show()
    
    description = generate_product_description(
        visual_analysis,
        product_info if product_info else None
    )
    
    print("\n" + "="*80)
    print("✨ ÜRÜN AÇIKLAMASI")
    print("="*80)
    print(f"\n{description}\n")
    
    return description

In [ ]:
# Hızlı test
# desc = quick_describe(
#     "product.jpg",
#     Kategori="kazak",
#     Malzeme="%100 pamuk",
#     Renk="kahverengi"
# )